In [0]:

#  This notebook generates new simulated data based on previous information

import json
import uuid
import random
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType
import numpy as np

TARGET_TABLE = "retail_lakehouse_dr.bronze.api_snapshots"
RUN_ID = str(uuid.uuid4())
EXTRACTED_AT = datetime.now(timezone.utc).isoformat()

SYNTHETIC_CARTS_PER_RUN = np.random.randint(200, 401)

bronze_schema = StructType([
    StructField("source", StringType(), False),
    StructField("record_id", StringType(), False),
    StructField("run_id", StringType(), False),
    StructField("extracted_at", StringType(), False),
    StructField("payload_json", StringType(), False)
])

def make_bronze_row(source, record_id, payload):
    return (
        source,
        str(record_id),
        RUN_ID,
        EXTRACTED_AT,
        json.dumps(
            payload,
            ensure_ascii=False,
            separators=(",", ":")
        )
    )

print(f"Generator Run ID: {RUN_ID}")
print(f"Extraction timestamp: {EXTRACTED_AT}")

Generator Run ID: a328a544-d9eb-4dd0-9abe-45ffa3a8be75
Extraction timestamp: 2026-10-25T17:10:18.482742+00:00


In [0]:
latest_products = spark.sql(f"""
    WITH ranked AS (
        SELECT
            record_id,
            payload_json,
            ROW_NUMBER() OVER (
                PARTITION BY record_id
                ORDER BY extracted_at DESC, run_id DESC
            ) AS row_num
        FROM {TARGET_TABLE}
        WHERE source = 'products_simulated'
    )
    SELECT record_id, payload_json
    FROM ranked
    WHERE row_num = 1
""").collect()

product_pool = [
    json.loads(row["payload_json"])
    for row in latest_products
]

if not product_pool:
    raise ValueError(
        "No simulated products found in Bronze. "
        "Run 01_bronze_ingestion successfully first."
    )

print(f"Latest simulated products loaded: {len(product_pool)}")

Latest simulated products loaded: 194


In [0]:
synthetic_carts = []
bronze_rows = []

for cart_number in range(SYNTHETIC_CARTS_PER_RUN):
    selected_count = random.randint(
        1,
        min(5, len(product_pool))
    )
    selected_products = random.sample(
        product_pool,
        selected_count
    )

    cart_products = []
    total = 0.0
    discounted_total = 0.0
    total_quantity = 0

    for product in selected_products:
        quantity = random.randint(1, 4)
        price = float(product["price"])
        discount = float(product.get("discountPercentage", 0))

        line_total = round(price * quantity, 2)
        line_discounted_total = round(
            price * (1 - discount / 100) * quantity,
            2
        )

        cart_products.append({
            "productId": int(product["id"]),
            "title": product["title"],
            "price": price,
            "quantity": quantity,
            "total": line_total,
            "discountPercentage": discount,
            "discountedTotal": line_discounted_total
        })

        total += line_total
        discounted_total += line_discounted_total
        total_quantity += quantity

    cart_id = f"sim_{RUN_ID[:8]}_{cart_number + 1}"

    synthetic_cart = {
        "id": cart_id,
        "userId": random.randint(1, 208),
        "products": cart_products,
        "total": round(total, 2),
        "discountedTotal": round(discounted_total, 2),
        "totalProducts": len(cart_products),
        "totalQuantity": total_quantity,
        "_simulation": {
            "run_id": RUN_ID,
            "created_at": EXTRACTED_AT,
            "origin": "synthetic_cart_generator"
        }
    }

    synthetic_carts.append(synthetic_cart)

    bronze_rows.append(
        make_bronze_row(
            "carts_simulated",
            cart_id,
            synthetic_cart
        )
    )

print(f"Synthetic carts generated: {len(synthetic_carts)}")

bronze_df = spark.createDataFrame(
    bronze_rows,
    schema=bronze_schema
)

display(bronze_df)

Synthetic carts generated: 5


source,record_id,run_id,extracted_at,payload_json
carts_simulated,sim_a328a544_1,a328a544-d9eb-4dd0-9abe-45ffa3a8be75,2026-10-25T17:10:18.482742+00:00,"{""id"":""sim_a328a544_1"",""userId"":194,""products"":[{""productId"":120,""title"":""Vaseline Men Body and Face Lotion"",""price"":9.15,""quantity"":4,""total"":36.6,""discountPercentage"":13.39,""discountedTotal"":31.7},{""productId"":45,""title"":""House Showpiece Plant"",""price"":37.03,""quantity"":4,""total"":148.12,""discountPercentage"":7.46,""discountedTotal"":137.07},{""productId"":188,""title"":""Pampi Shoes"",""price"":29.65,""quantity"":4,""total"":118.6,""discountPercentage"":14.14,""discountedTotal"":101.83}],""total"":303.32,""discountedTotal"":270.6,""totalProducts"":3,""totalQuantity"":12,""_simulation"":{""run_id"":""a328a544-d9eb-4dd0-9abe-45ffa3a8be75"",""created_at"":""2026-10-25T17:10:18.482742+00:00"",""origin"":""synthetic_cart_generator""}}"
carts_simulated,sim_a328a544_2,a328a544-d9eb-4dd0-9abe-45ffa3a8be75,2026-10-25T17:10:18.482742+00:00,"{""id"":""sim_a328a544_2"",""userId"":79,""products"":[{""productId"":31,""title"":""Lemon"",""price"":0.84,""quantity"":2,""total"":1.68,""discountPercentage"":9.7,""discountedTotal"":1.52},{""productId"":122,""title"":""iPhone 6"",""price"":295.9,""quantity"":3,""total"":887.7,""discountPercentage"":6.69,""discountedTotal"":828.31}],""total"":889.38,""discountedTotal"":829.83,""totalProducts"":2,""totalQuantity"":5,""_simulation"":{""run_id"":""a328a544-d9eb-4dd0-9abe-45ffa3a8be75"",""created_at"":""2026-10-25T17:10:18.482742+00:00"",""origin"":""synthetic_cart_generator""}}"
carts_simulated,sim_a328a544_3,a328a544-d9eb-4dd0-9abe-45ffa3a8be75,2026-10-25T17:10:18.482742+00:00,"{""id"":""sim_a328a544_3"",""userId"":144,""products"":[{""productId"":14,""title"":""Knoll Saarinen Executive Conference Chair"",""price"":499.85,""quantity"":4,""total"":1999.4,""discountPercentage"":2.01,""discountedTotal"":1959.21},{""productId"":49,""title"":""Black Aluminium Cup"",""price"":4.88,""quantity"":2,""total"":9.76,""discountPercentage"":15.65,""discountedTotal"":8.23},{""productId"":176,""title"":""Women Handbag Black"",""price"":70.84,""quantity"":1,""total"":70.84,""discountPercentage"":11.63,""discountedTotal"":62.6},{""productId"":72,""title"":""Slotted Turner"",""price"":8.98,""quantity"":4,""total"":35.92,""discountPercentage"":13.35,""discountedTotal"":31.12}],""total"":2115.92,""discountedTotal"":2061.16,""totalProducts"":4,""totalQuantity"":11,""_simulation"":{""run_id"":""a328a544-d9eb-4dd0-9abe-45ffa3a8be75"",""created_at"":""2026-10-25T17:10:18.482742+00:00"",""origin"":""synthetic_cart_generator""}}"
carts_simulated,sim_a328a544_4,a328a544-d9eb-4dd0-9abe-45ffa3a8be75,2026-10-25T17:10:18.482742+00:00,"{""id"":""sim_a328a544_4"",""userId"":12,""products"":[{""productId"":30,""title"":""Kiwi"",""price"":2.72,""quantity"":3,""total"":8.16,""discountPercentage"":15.22,""discountedTotal"":6.92},{""productId"":114,""title"":""Kawasaki Z800"",""price"":10334.68,""quantity"":2,""total"":20669.36,""discountPercentage"":9.77,""discountedTotal"":18649.96},{""productId"":20,""title"":""Cooking Oil"",""price"":5.22,""quantity"":3,""total"":15.66,""discountPercentage"":9.33,""discountedTotal"":14.2}],""total"":20693.18,""discountedTotal"":18671.08,""totalProducts"":3,""totalQuantity"":8,""_simulation"":{""run_id"":""a328a544-d9eb-4dd0-9abe-45ffa3a8be75"",""created_at"":""2026-10-25T17:10:18.482742+00:00"",""origin"":""synthetic_cart_generator""}}"
carts_simulated,sim_a328a544_5,a328a544-d9eb-4dd0-9abe-45ffa3a8be75,2026-10-25T17:10:18.482742+00:00,"{""id"":""sim_a328a544_5"",""userId"":45,""products"":[{""productId"":59,""title"":""Glass"",""price"":6.06,""quantity"":4,""total"":24.24,""discountPercentage"":7.92,""discountedTotal"":22.32},{""productId"":175,""title"":""White Faux Leather Backpack"",""price"":46.86,""quantity"":3,""total"":140.58,""discountPercentage"":15.2,""discountedTotal"":119.21}],""total"":164.82,""discou

In [0]:
if len(synthetic_carts) != SYNTHETIC_CARTS_PER_RUN:
    raise ValueError(
        f"Expected {SYNTHETIC_CARTS_PER_RUN} carts, "
        f"but generated {len(synthetic_carts)}."
    )

existing_ids = spark.sql(f"""
    SELECT record_id
    FROM {TARGET_TABLE}
    WHERE source = 'carts_simulated'
      AND record_id IN (
          {", ".join("'" + cart["id"] + "'" for cart in synthetic_carts)}
      )
""").collect()

if existing_ids:
    raise ValueError(
        f"Cart IDs already exist in Bronze: {existing_ids}"
    )

(bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(TARGET_TABLE))

print(f"Successfully appended {len(synthetic_carts)} synthetic carts.")

Successfully appended 5 synthetic carts.


In [0]:
display(spark.sql(f"""
    SELECT
        source,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT record_id) AS distinct_carts,
        COUNT(DISTINCT run_id) AS total_runs,
        MAX(extracted_at) AS latest_extraction
    FROM {TARGET_TABLE}
    WHERE source = 'carts_simulated'
    GROUP BY source
"""))

display(spark.sql(f"""
    SELECT
        record_id,
        run_id,
        extracted_at,
        get_json_object(payload_json, '$.total') AS total,
        get_json_object(
            payload_json, '$.totalProducts'
        ) AS total_products
    FROM {TARGET_TABLE}
    WHERE source = 'carts_simulated'
      AND run_id = '{RUN_ID}'
    ORDER BY record_id
"""))

source,total_rows,distinct_carts,total_runs,latest_extraction
carts_simulated,485,480,96,2026-10-25T17:10:18.482742+00:00


record_id,run_id,extracted_at,total,total_products
sim_a328a544_1,a328a544-d9eb-4dd0-9abe-45ffa3a8be75,2026-10-25T17:10:18.482742+00:00,303.32,3
sim_a328a544_2,a328a544-d9eb-4dd0-9abe-45ffa3a8be75,2026-10-25T17:10:18.482742+00:00,889.38,2
sim_a328a544_3,a328a544-d9eb-4dd0-9abe-45ffa3a8be75,2026-10-25T17:10:18.482742+00:00,2115.92,4
sim_a328a544_4,a328a544-d9eb-4dd0-9abe-45ffa3a8be75,2026-10-25T17:10:18.482742+00:00,20693.18,3
sim_a328a544_5,a328a544-d9eb-4dd0-9abe-45ffa3a8be75,2026-10-25T17:10:18.482742+00:00,164.82,2
